# Phase 1.3：批量解析、验证和交付 chunks.json

## 目标

把前两课的理解接入项目生产模块：批量读取输入目录、分块、生成稳定 ID、验证数据契约、保存并重新加载 JSON。

**本课交付：** `data/processed/chunks.json`，它是 Phase 2 的唯一输入。

## Evidence Quest 任务卡：Phase 1.3：证据流水线交付

**你的身份：** 知识库流水线负责人  
**案件背景：** 调查组不能手工处理每一份文件。你要把解析、切分、稳定 ID 和 JSON 交付串成可重复执行的流水线。

### 本关专业 Goal

从输入目录生成下游检索可以直接读取的 chunks.json。

### 你要交付的作品

**可复现的 Phase 1 文档考古工具**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：流水线交付官  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. 生产函数解决了哪些重复工作？

手写版本帮助理解原理，但正式项目还需要处理文件排序、PDF 页码、稳定哈希、输出格式和异常边界。`build_chunks()` 把这些工作集中起来，并由测试保护。

使用模块不是停止学习，而是把“已经理解的算法”放进可复用工程边界。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase1.3'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase1.3
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 从生产模块导入批量构建函数。
from phase1_doc_parser.main import build_chunks

# 从生产模块导入正式的递归分块器。
from phase1_doc_parser.splitter import RecursiveSplitter

# 指定真实输入目录。
input_directory = ROOT / "phase1_doc_parser" / "examples" / "input"

# 选择本阶段经过实验的分块参数。
splitter = RecursiveSplitter(chunk_size=128, overlap=32)

# 批量生成结构化 Chunk。
chunks = build_chunks(input_directory, splitter)

# 打印总数量，确认批处理确实产生结果。
print("生成 Chunk 数:", len(chunks))

# 查看第一条完整记录，确认字段没有在批处理中丢失。
print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

# 不能为空，否则后续检索没有任何数据。
assert chunks

生成 Chunk 数: 4
{
  "id": "3692b05e025373a8",
  "text": "# Phase 1 Quickstart\n\n文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。\n\n## Chunk 策略",
  "source": "D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\quickstart.md",
  "page": null,
  "chunk_index": 0,
  "metadata": {
    "format": "markdown",
    "headings": [
      "Phase 1 Quickstart",
      "Chunk 策略"
    ]
  }
}


## 2. 逐字段验证数据契约

数据契约不是形式主义。`id` 用于评估和引用，`text` 用于检索，`source/page` 用于回溯，`metadata` 用于解释。任何字段缺失，最终 API 都可能无法给用户可靠证据。

In [4]:
# 定义每条 Chunk 必须拥有的字段集合。
required_fields = {"id", "text", "source", "page", "chunk_index", "metadata"}

# 逐条检查字段集合和非空字段。
for chunk in chunks:
    # 确认当前记录包含全部字段。
    assert required_fields <= chunk.keys()

    # 确认正文、ID 和来源不能为空。
    assert chunk["id"] and chunk["text"] and chunk["source"]

# 把全部 Chunk ID 放进列表，准备检查重复。
chunk_ids = [str(chunk["id"]) for chunk in chunks]

# 稳定 ID 必须唯一，否则两个证据无法区分。
assert len(chunk_ids) == len(set(chunk_ids))

# 输出契约检查结果。
print("字段、非空值和唯一 ID 检查通过。")

字段、非空值和唯一 ID 检查通过。


In [5]:
# 逐条检查正文长度不超过分块器的上限。
for chunk in chunks:
    # 当前长度必须小于或等于 128 个字符。
    assert len(str(chunk["text"])) <= 128

# 检查同一输入重复运行能得到相同 ID 顺序。
repeated_chunks = build_chunks(input_directory, RecursiveSplitter(chunk_size=128, overlap=32))

# 提取第二次运行的 ID 列表。
repeated_ids = [str(chunk["id"]) for chunk in repeated_chunks]

# 断言两次运行完全一致，证明索引引用可复现。
assert chunk_ids == repeated_ids

# 输出稳定性检查结果。
print("长度和稳定 ID 检查通过。")

长度和稳定 ID 检查通过。


## 3. 保存和重新加载：磁盘文件才是阶段交付

内存中的 `chunks` 只在当前 Kernel 存活。下一阶段或服务启动时需要从磁盘读取，所以保存后必须重新加载并验证，而不是只看写入函数没有报错。

In [6]:
# 创建处理数据目录。
processed_directory = ROOT / "data" / "processed"
processed_directory.mkdir(parents=True, exist_ok=True)

# 指定 Phase 1 的正式交付文件。
chunks_path = processed_directory / "chunks.json"

# 以 UTF-8 JSON 保存全部 Chunk。
chunks_path.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")

# 从磁盘重新读取 JSON，模拟下一阶段的输入。
loaded_chunks = json.loads(chunks_path.read_text(encoding="utf-8"))

# 确认重新加载后的数量和 ID 与内存结果一致。
assert len(loaded_chunks) == len(chunks)
assert [item["id"] for item in loaded_chunks] == chunk_ids

# 打印最终交付路径和数量。
print("Phase 1 交付:", chunks_path, "chunks=", len(loaded_chunks))

Phase 1 交付: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\chunks.json chunks= 4


## 4. 追溯演练：从 Chunk 回到原文

随机选择一条 Chunk，打印它的来源、页码和正文。这个动作模拟最终用户点击 citation 的第一步，也是判断 Phase 1 是否真正完成的关键。

In [7]:
# 选择第一条 Chunk 作为追溯样本。
trace_sample = loaded_chunks[0]

# 输出引用所需的最小信息。
print("chunk_id:", trace_sample["id"])
print("source:", trace_sample["source"])
print("page:", trace_sample["page"])
print("text:", trace_sample["text"])

# 确认来源路径仍然指向存在的原始文件。
assert Path(trace_sample["source"]).is_file()

chunk_id: 3692b05e025373a8
source: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input\quickstart.md
page: None
text: # Phase 1 Quickstart

文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。

## Chunk 策略


## Phase 1 最终闸门

- [ ] 能从原始文件解释到 `ParsedDocument`、Chunk、稳定 ID 的变化过程。
- [ ] 所有 Chunk 通过字段、非空、长度和唯一性检查。
- [ ] 重复运行得到相同 ID。
- [ ] `chunks.json` 能保存、重新加载并追溯到真实文件。

现在 Phase 2 可以把这份文件当作稳定语料，而不必重新猜测文档边界。

## Boss Challenge：把输入目录换成你准备的一份 Markdown，检查稳定 chunk_id 是否在两次运行中一致。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [8]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [9]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/chunks.json', 'phase1_doc_parser/output/chunks.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\chunks.json', 'phase1_doc_parser\\output\\chunks.json']
